In [1]:

"""
Project: Trading Bot Using Indicators of choice

Goal:
The goal of this project is to build a trading bot that will show good opportunites to buy stock based on certain indicators. The indicators
chosen are : Stochastic RSI, RSI (Relative Strength Index), Moving Average (200 day). These will be viewed from the weekly chart. 
With this tool we would be able to scrape stocks for potential setups that mirror what I would look for in a trade setup.
The strategy will then be backtested to show its success rate. The trading bot itself will then point out if a stock is 
a "buy" depending on whether indicators are showing the correct parameters

Features:
- Stock indicator check
- Backtesting tool
- Multi-stock scanner

Status: Working on inputing the strategy (based on my actual trading strategy)
"""

import pandas as pd
import numpy as np
import yfinance as yf

In [2]:
# Strategy Settings

'''
Strategy Rules (all of these are on the weekly timeframe):

Buy the stock if:
- RSI > 30
- StochRSI < 20 (both K and D lines)
- price is > 200 day moving average

Otherwise, the stock is not a Buy
'''
# Thresholds for the indicators

RSI_Threshold = 30
SMA_Period = 200
StochRSI_Threshold = 20

In [3]:
# Stock(s) that will be used to test
ticker = "IBIT"

df = yf.download(ticker, period='5y', interval='1wk')

[*********************100%***********************]  1 of 1 completed


In [4]:
# Indicator Functions

#RSI Formula uses Wilder smoothing, math to make it similar to TradingView's RSI
def calculate_rsi(close, period=14):
    delta = close.diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(alpha=1/period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, adjust=False).mean()

    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))

    return rsi

# K and D periods for RSI both resemble those of TradingView
def calculate_stoch_rsi(close, rsi_period=14, stoch_period=14, k_period=3, d_period=3):
    rsi = calculate_rsi(close, rsi_period)

    min_rsi = rsi.rolling(stoch_period).min()
    max_rsi = rsi.rolling(stoch_period).max()

    stoch_rsi = (rsi - min_rsi) / (max_rsi - min_rsi)

    k = stoch_rsi.rolling(k_period).mean()
    d = k.rolling(d_period).mean()

    return {'k': k, 'd': d}


def calculate_sma(close, period=200):
    sma = close.rolling(window=period).mean()

    return sma
    

In [5]:
# Analyze Indicator Functions

# closing price of the stock and returning the last RSI
close = df['Close'][ticker]

# calling the RSI
rsi = calculate_rsi(close, 14)

last_rsi = rsi.iloc[-1]

def analyze_rsi(last_rsi):

    if pd.isna(last_rsi):
        return 'No RSI data available'
        
    if last_rsi < 20:
        return 'This stock is way oversold'
    elif last_rsi > 20 and last_rsi < 50:
        return 'Stock is leaning oversold'
    elif last_rsi < 70 and last_rsi >= 50:
        return 'Stock is leaning overbought'
    elif last_rsi >= 70:
        return 'This stock is overbought for sure.'

rsi_analysis = analyze_rsi(last_rsi)

# calling the StochRSI (k and d lines)

stoch_rsi = calculate_stoch_rsi(close)

stoch_k = stoch_rsi['k'].iloc[-1]
stoch_d = stoch_rsi['d'].iloc[-1]

def analyze_stoch(stoch_k, stoch_d):

    if pd.isna(stoch_k) or pd.isna(stoch_d):
        return 'No StochRSI data available'
        
    if stoch_k <= .20 and stoch_d <= .20:
        return 'Momentum is bearish'
    elif stoch_k >= .70 and stoch_d >= .70:
        return 'Full bullish momentum'
    elif stoch_k >= .70 and stoch_d <= .70: # k line can lead before d fully confirms the bullishness
        return 'StochRSI is showing bullish momentum'
    else:
        return 'StochRSI in midrange, evaluate further'

stoch_analysis = analyze_stoch(stoch_k, stoch_d)

# calling the 200SMA 

sma_real = calculate_sma(close)

sma_value = sma_real.iloc[-1]

def analyze_sma(close, sma_value):

    if pd.isna(sma_value): # edge case to check if there is any value in there.
        return 'No SMA(200) data available'

    if close.iloc[-1] >= sma_value:
        return 'This stock is bull market bullish'
    else:
        return 'Stock may be in a bear trend.'

sma_analysis = analyze_sma(close, sma_value) # commentary on the SMA value relative to price


In [6]:
# Multi-Stock Scanner

def analyze_ticker(ticker):
    
    df = yf.download(ticker, period='5y', interval='1wk') # pulling data from yfinance
    
    close = df['Close'][ticker] # Extracting closing price series

    # ---- RSI ----
    rsi = calculate_rsi(close, 14)
    last_rsi = rsi.iloc[-1]
    rsi_analysis = analyze_rsi(last_rsi)

    # ---- StochRSI ----
    stoch_rsi = calculate_stoch_rsi(close)
    stoch_k = stoch_rsi['k'].iloc[-1]
    stoch_d = stoch_rsi['d'].iloc[-1]
    stoch_analysis = analyze_stoch(stoch_k, stoch_d)

    # ---- SMA(200) ----
    sma_real = calculate_sma(close)
    sma_value = sma_real.iloc[-1]
    sma_analysis = analyze_sma(close, sma_value)

    # ---- Creating the row ----
    return {
    'Ticker': ticker,
    'Price': round(close.iloc[-1], 2),
    'RSI': round(last_rsi, 2),
    'RSI_Comment': rsi_analysis,
    'StochRSI': f'K={stoch_k:.2f}, D={stoch_d:.2f}',
    'StochRSI_Comment': stoch_analysis,
    'SMA(200)': round(sma_value, 2),
    'SMA_Comment': sma_analysis
    }

# calling the functions and creating the table

tickers = ['AAPL', 'MSFT', 'META', 'CRM', 'IBIT']

rows = []

for ticker in tickers:
    result = analyze_ticker(ticker)
    rows.append(result)

scanner_table = pd.DataFrame(rows)

scanner_table

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


,Ticker,Price,RSI,RSI_Comment,StochRSI,StochRSI_Comment,SMA(200),SMA_Comment
0,AAPL,300.23,69.72,Stock is leaning overbought,"K=1.00, D=0.91",Full bullish momentum,201.99,This stock is bull market bullish
1,MSFT,421.92,48.53,Stock is leaning oversold,"K=0.93, D=0.94",Full bullish momentum,378.57,This stock is bull market bullish
2,META,614.23,46.81,Stock is leaning oversold,"K=0.59, D=0.68","StochRSI in midrange, evaluate further",452.12,This stock is bull market bullish
3,CRM,173.51,37.30,Stock is leaning oversold,"K=0.88, D=0.87",Full bullish momentum,233.04,Stock may be in a bear trend.
4,IBIT,44.82,47.31,Stock is leaning oversold,"K=0.98, D=0.99",Full bullish momentum,NaN,No SMA(200) data available


In [7]:
# Some test scripts here

# Notes
# Clean notebook flow
# Remove old duplicate single-ticker cells. Keep only the real pipeline.
# Wrap scanner into run_scanner(tickers)
# So you can call one clean function and get the table.
# Add error handling
# Bad tickers, missing data, NaNs, weird Yahoo nonsense.
# Add a bullish score
# Example: RSI + Stoch + SMA combine into a score like 7/10.
# Backtester
# Test: “When this setup appeared, what was the 3-month return?”

In [8]:
# Signal Ticker Test

In [9]:
# Backtester

In [10]:
# Results / Charts